# WP set encoder — Kaggle GPU trainingFree path for training a win-probability model (PLAN.md Phase 4+). 30 GPU-hours/week, no balanceto manage. A run of this size takes **2–3 minutes**, versus ~45 on the laptop.**Before running:**1. Locally: `scripts/cloud/pack_training.sh reg_mc wp-v1` and unpack it, or just take the directory   `data/features/reg_mc/wp-v1` (~110 MB).2. Add it as a **private Kaggle Dataset** (Datasets → New Dataset → upload the folder). Name it   `vgc-wp-features` so the path below matches, or edit `DATA` in the next cell.3. Notebook settings: **Accelerator = GPU T4 x2** (or P100), **Internet off** is fine — nothing is   downloaded.4. Add this notebook's `set_torch.py`: upload `src/vgc/wp/set_torch.py` as a second Dataset (or paste   it into a cell). `SRC` below points at it.**After running:** download `model.onnx` + `train.json` from the output, put them in`models/wp/reg_mc/<version>/`, then locally run `vgc wp card`, `vgc wp calibrate` and `vgc wp eval`.

In [ ]:
import json, os, shutil, subprocess, sys, timefrom pathlib import PathDATA = Path("/kaggle/input/vgc-wp-features/wp-v1")   # the feature dataset directorySRC  = Path("/kaggle/input/vgc-set-torch")           # directory holding set_torch.pyOUT  = Path("/kaggle/working/model")pkg = Path("/kaggle/working/src/vgc/wp")pkg.mkdir(parents=True, exist_ok=True)for p in (pkg.parent.parent, pkg.parent, pkg):    (p / "__init__.py").touch()shutil.copy(SRC / "set_torch.py", pkg / "set_torch.py")import torchprint("torch", torch.__version__, "| cuda", torch.cuda.is_available(),      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")print("dataset:", json.loads((DATA / "info.json").read_text())["rows"])

## One runFlags mirror `scripts/cloud/run_training.sh`. On a GPU, use a larger batch (`--bs 2048`): the modelis small and small batches leave the GPU idle.

In [ ]:
def train(name, *flags, epochs=12, bs=2048):    out = OUT / name    cmd = [sys.executable, "-u", "-m", "vgc.wp.set_torch", "--data", str(DATA), "--out", str(out),           "--device", "auto", "--epochs", str(epochs), "--bs", str(bs), *map(str, flags)]    t0 = time.time()    subprocess.run(cmd, cwd="/kaggle/working", env={**os.environ, "PYTHONPATH": "/kaggle/working/src"}, check=True)    r = json.loads((out / "train.json").read_text())    print(f"{name}: val {r['val_wp_logloss']} (calibrated {r['val_wp_logloss_calibrated']}), "          f"best epoch {r['best_epoch']}, {r['parameters']} params, {time.time()-t0:.0f}s")    return rtrain("wp-v1-set", "--d", 128, "--layers", 3, "--dropout", 0.2, "--weight-decay", 0.1, "--id-dropout", 0.3)

## A sweepThe real reason to use a GPU: try several configurations and keep the best by validation log loss.Snapshots inside one battle share an outcome, so these models memorise battles easily — the knobsthat matter are size (`--d`, `--layers`), `--dropout`, `--weight-decay` and `--id-dropout`.

In [ ]:
configs = {    "small":      ("--d", 64,  "--layers", 2, "--dropout", 0.3, "--weight-decay", 0.1, "--id-dropout", 0.2),    "mid-id0.3":  ("--d", 128, "--layers", 3, "--dropout", 0.2, "--weight-decay", 0.1, "--id-dropout", 0.3),    "mid-id0.5":  ("--d", 128, "--layers", 3, "--dropout", 0.2, "--weight-decay", 0.1, "--id-dropout", 0.5),    "wide":       ("--d", 192, "--layers", 3, "--dropout", 0.3, "--weight-decay", 0.2, "--id-dropout", 0.4),}results = {name: train(name, *flags) for name, flags in configs.items()}for name, r in sorted(results.items(), key=lambda kv: kv[1]["val_wp_logloss_calibrated"]):    print(f"{name:12} {r['val_wp_logloss_calibrated']:.5f}  (epoch {r['best_epoch']}, {r['parameters']} params)")

## CollectDownload the zip, unpack into `models/wp/reg_mc/`, then locally:```bashvgc wp card      --version <name>vgc wp calibrate --version <name>vgc wp eval      --version <name> --baseline wp-v1-gbt --baseline wp-v1-logistic --baseline constant```Evaluation always runs locally against the frozen held-out split — nothing here decides that.

In [ ]:
shutil.make_archive("/kaggle/working/wp-models", "zip", OUT)print(sorted(p.name for p in OUT.glob("*")))